# Computational performance analysis - Logistic Regression

Initial environment configuration

In [1]:
import os
import sys

# Ensure custom capymoa wrapper is loaded over the installed one
wrapper_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src"))
if wrapper_path not in sys.path:
    sys.path.insert(0, wrapper_path)

os.environ["CAPYMOA_MOA_JAR"] = os.path.abspath(os.path.join(os.getcwd(), "..", "custom_moa_full.jar"))

Global variables

In [2]:
DEFAULT_LR = 0.01
DEFAULT_BIAS_LR = 0.01
DEFAULT_L1 = 0.0
DEFAULT_L2 = 0.0
DEFAULT_CLIP = 1e12
DEFAULT_BIAS_INIT = 0.0
MAX_INSTANCES = 100000
SEED = 42

Global imports

In [3]:
from river import linear_model, optim, evaluate, metrics
from capymoa.evaluation import prequential_evaluation
from tabulate import tabulate
import importlib.util
import sys
import os
import time
import tracemalloc

Dynamically load the custom LogisticRegression over the installed capymoa package

In [4]:
file_path = os.path.abspath(os.path.join(os.getcwd(), "..", "capymoa", "src", "capymoa", "classifier", "_logistic_regression.py"))
spec = importlib.util.spec_from_file_location("capymoa.classifier._logistic_regression", file_path)
module = importlib.util.module_from_spec(spec)
sys.modules["capymoa.classifier._logistic_regression"] = module
spec.loader.exec_module(module)
LogisticRegression = module.LogisticRegression

Global functions

In [5]:
def adaptStreamForRiver(stream):
    data = []

    for i, instance in enumerate(stream):
        if(i > MAX_INSTANCES): break

        # features
        x = {f"f{j}": float(v) for j, v in enumerate(instance.x)}

        # label
        y = instance.y_index
        
        data.append((x, y))
    
    return data

In [6]:
def evaluateStream(stream_factory, lr=DEFAULT_LR, b_lr=DEFAULT_BIAS_LR, l1=DEFAULT_L1, l2=DEFAULT_L2, clip=DEFAULT_CLIP, bias_init=DEFAULT_BIAS_INIT):
    # stream_factory must be deterministic. It is the constructor of the stream.

    capyMoaResults = _evaluateStreamOnCapyMoa(stream_factory(), lr, b_lr, l1, l2, clip, bias_init)

    riverStream = adaptStreamForRiver(stream_factory())

    riverResults = _evaluateStreamOnRiver(riverStream, lr, b_lr, l1, l2, clip, bias_init)

    table = []
    for key in ["Time (s)", "Memory (MB)"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.4f}",
            f"{river:.4f}",
            f"{(capy - river):+.4f}"
        ])

    for key in ["Accuracy", "F1"]:
        capy = capyMoaResults[key]
        river = riverResults[key]

        table.append([
            key,
            f"{capy:.2f}%",
            f"{river:.2f}%",
            f"{(capy - river):+.2f}%"
        ])

    print("\n--- Comparison CapyMOA vs River ---\n")
    print(
        tabulate(
            table,
            headers=["Metric", "CapyMOA", "River", "Delta"],
            tablefmt="fancy_grid"
        )
    )

def _evaluateStreamOnCapyMoa(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_capymoa = LogisticRegression(
        schema=stream.get_schema(),
        learning_rate=lr,
        bias_learning_rate=b_lr,
        l1_penalty=l1,
        l2_penalty=l2,
        clip_gradient=clip,
        bias_init=bias_init
    )

    tracemalloc.start()
    start_time = time.time()

    # prequential evaluation using CapyMOA built-in function
    results = prequential_evaluation(
        stream=stream,
        learner=log_reg_capymoa,
        max_instances=MAX_INSTANCES
    )

    end_time = time.time()
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    metrics = {
        "Accuracy": results['cumulative'].accuracy(),
        "F1": results['cumulative'].f1_score(),
        "Time (s)": end_time - start_time,
        "Memory (MB)": peak_memory / (1024 * 1024)
    }

    return metrics

def _evaluateStreamOnRiver(stream, lr, b_lr, l1, l2, clip, bias_init):
    log_reg_river = linear_model.LogisticRegression(
        optimizer=optim.SGD(lr),
        intercept_lr=b_lr,
        l1=l1,
        l2=l2,
        clip_gradient=clip,
        intercept_init=bias_init
    )

    metric = (
        metrics.Accuracy() +
        metrics.F1()
    )

    tracemalloc.start()
    start_time = time.time()

    # prequential evaluation using River built-in function
    result = evaluate.progressive_val_score(
        dataset=stream,
        model=log_reg_river,
        metric=metric
    )

    end_time = time.time()
    _, peak_memory = tracemalloc.get_traced_memory()
    tracemalloc.stop()

    results = {}
    results["Accuracy"] = float(result[0].get()*100)
    results["F1"] = float(result[1].get()*100)

    results["Time (s)"] = end_time - start_time
    results["Memory (MB)"] = peak_memory / (1024 * 1024)

    return results

## Electricity dataset

In [7]:
from capymoa.datasets import Electricity

evaluateStream(Electricity)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.3281    │ 3.8026  │ -3.4746 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0432    │ 0.1839  │ -0.1407 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 69.93%    │ 69.93%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 68.83%    │ 75.02%  │ -6.19%  │
╘═════════════╧═══════════╧═════════╧═════════╛


## ElectricityTiny dataset

In [8]:
from capymoa.datasets import ElectricityTiny

evaluateStream(ElectricityTiny)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.0127    │ 0.1534  │ -0.1406 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0026    │ 0.1484  │ -0.1458 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 62.85%    │ 62.85%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 58.67%    │ 38.85%  │ +19.82% │
╘═════════════╧═══════════╧═════════╧═════════╛


## RandomRBFGenerator

In [9]:
from capymoa.stream.generator import RandomRBFGenerator

def make_stream():
    return RandomRBFGenerator(
        number_of_classes=2,
        number_of_attributes=50,
        number_of_centroids=100,
        model_random_seed=SEED
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤══════════╕
│ Metric      │ CapyMOA   │ River   │ Delta    │
╞═════════════╪═══════════╪═════════╪══════════╡
│ Time (s)    │ 0.2783    │ 16.4435 │ -16.1652 │
├─────────────┼───────────┼─────────┼──────────┤
│ Memory (MB) │ 0.1523    │ 0.1560  │ -0.0037  │
├─────────────┼───────────┼─────────┼──────────┤
│ Accuracy    │ 85.28%    │ 85.28%  │ +0.00%   │
├─────────────┼───────────┼─────────┼──────────┤
│ F1          │ 84.75%    │ 81.59%  │ +3.16%   │
╘═════════════╧═══════════╧═════════╧══════════╛


## Hyper100k dataset

In [10]:
from capymoa.datasets import Hyper100k

evaluateStream(Hyper100k)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.6759    │ 8.7101  │ -8.0342 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.1519    │ 0.1494  │ +0.0024 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 89.86%    │ 89.86%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 89.92%    │ 90.15%  │ -0.23%  │
╘═════════════╧═══════════╧═════════╧═════════╛


## SEA dataset generator

In [11]:
from capymoa.stream.generator import SEA

def make_stream():
    return SEA(
        instance_random_seed=SEED,
        function=1,
        balance_classes=False,
        noise_percentage=10,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.0385    │ 7.0492  │ -7.0106 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0108    │ 0.1573  │ -0.1465 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 82.91%    │ 82.91%  │ -0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 81.09%    │ 87.05%  │ -5.96%  │
╘═════════════╧═══════════╧═════════╧═════════╛


## HyperPlaneClassification dataset

In [12]:
from capymoa.stream.generator import HyperPlaneClassification

def make_stream():
    return HyperPlaneClassification(
        instance_random_seed=SEED,
        number_of_classes=2,
        number_of_attributes=10,
        number_of_drifting_attributes=2,
        magnitude_of_change=0.0,
        noise_percentage=5,
        sigma_percentage=10,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.0460    │ 8.6839  │ -8.6379 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0108    │ 0.1497  │ -0.1388 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 89.87%    │ 89.87%  │ -0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 89.94%    │ 90.18%  │ -0.24%  │
╘═════════════╧═══════════╧═════════╧═════════╛


## RandomTreeGenerator

In [13]:
from capymoa.stream.generator import RandomTreeGenerator

def make_stream():
    return RandomTreeGenerator(
        instance_random_seed=SEED,
        tree_random_seed=SEED,
        num_classes=2,
        num_nominals=0,
        num_numerics=5,
        max_tree_depth=5,
        first_leaf_level=3,
        leaf_fraction=0.15,
    )

evaluateStream(make_stream)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.0982    │ 7.6126  │ -7.5143 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0110    │ 0.1579  │ -0.1469 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 81.05%    │ 81.06%  │ -0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 77.21%    │ 66.09%  │ +11.12% │
╘═════════════╧═══════════╧═════════╧═════════╛


## Electricity dataset (changed model parameters)

### L2

In [14]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l2=0.01)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.0937    │ 4.1623  │ -4.0687 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0062    │ 0.1493  │ -0.1431 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 68.49%    │ 68.49%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 67.21%    │ 74.33%  │ -7.12%  │
╘═════════════╧═══════════╧═════════╧═════════╛


### L1

In [15]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, l1=0.01)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.1000    │ 4.3153  │ -4.2153 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0062    │ 0.0093  │ -0.0031 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 65.76%    │ 65.76%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 64.06%    │ 73.34%  │ -9.28%  │
╘═════════════╧═══════════╧═════════╧═════════╛


### Learning rate

In [16]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, lr=0.1, b_lr=0.1)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.0937    │ 3.8539  │ -3.7602 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0062    │ 0.1492  │ -0.1431 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 81.08%    │ 81.08%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 80.57%    │ 83.76%  │ -3.19%  │
╘═════════════╧═══════════╧═════════╧═════════╛


### Gradient clipping

In [17]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, clip=1)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.0952    │ 3.7898  │ -3.6947 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0061    │ 0.0076  │ -0.0015 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 69.93%    │ 69.93%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 68.83%    │ 75.02%  │ -6.19%  │
╘═════════════╧═══════════╧═════════╧═════════╛


### Bias initialization

In [18]:
from capymoa.datasets import Electricity

evaluateStream(Electricity, bias_init=3)


--- Comparison CapyMOA vs River ---

╒═════════════╤═══════════╤═════════╤═════════╕
│ Metric      │ CapyMOA   │ River   │ Delta   │
╞═════════════╪═══════════╪═════════╪═════════╡
│ Time (s)    │ 0.1727    │ 3.6765  │ -3.5038 │
├─────────────┼───────────┼─────────┼─────────┤
│ Memory (MB) │ 0.0062    │ 0.1489  │ -0.1427 │
├─────────────┼───────────┼─────────┼─────────┤
│ Accuracy    │ 70.62%    │ 70.62%  │ +0.00%  │
├─────────────┼───────────┼─────────┼─────────┤
│ F1          │ 69.55%    │ 75.65%  │ -6.10%  │
╘═════════════╧═══════════╧═════════╧═════════╛
